In [1]:
import os
import re
import json
import fitz 
import asyncio
import pandas as pd
from glob import glob
from tqdm import tqdm
from pathlib import Path
from mistralai import Mistral
from dotenv import load_dotenv
from types import SimpleNamespace


load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]
client = Mistral(api_key=api_key)

In [2]:
def process_filename(file_path: str | Path, limit_folder=None) -> str:
    path_obj = Path(file_path).resolve()
    return f"\\\\?\\{path_obj}"

def upload_pdf_mistral(filepath: str):
    safe_path = process_filename(filepath)
    
    with open(safe_path, "rb") as f:
        uploaded_pdf = client.files.upload(
            file={
                "file_name": Path(filepath).name,
                "content": f,
            },
            purpose="ocr"
        )
    
    signed_url = client.files.get_signed_url(file_id=uploaded_pdf.id)
    return signed_url.url

def parse_pdf_text(filepath: str):
    safe_path = process_filename(filepath)
    
    try:
        with open(safe_path, "rb") as f:
            file_bytes = f.read()
            
        doc = fitz.open(stream=file_bytes, filetype="pdf")
        
        pages_list = []
        for page in doc:
            text_content = page.get_text("text") 
            
            page_obj = SimpleNamespace(markdown=text_content)
            pages_list.append(page_obj)
        
        return SimpleNamespace(pages=pages_list)
        
    except OSError as e:
        print(f"Error opening {Path(filepath).name}: {e}")
        return None

def parse_pdf_ocr(filepath: str):
    try:
        ocr_response = client.ocr.process(
            model="mistral-ocr-latest",
            document={
                "type": "document_url",
                "document_url": upload_pdf_mistral(filepath=filepath),
            },
            include_image_base64=True,
        )
        return ocr_response
    except Exception as e:
        print(f"Error OCR-ing {filepath}: {e}")
        return None

def parse_pdf(filepath: str, method: str = "ocr"):
    if method == "ocr":
        return parse_pdf_ocr(filepath)
    elif method == "text":
        return parse_pdf_text(filepath)
    else:
        raise ValueError("Method must be 'ocr' or 'text'")


In [3]:
statute_path = r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\statute\kolonial_kuh_perdata_fix.pdf"
responses = parse_pdf(statute_path, method="text")
responses

namespace(pages=[namespace(markdown='KITAB UNDANG-UNDANG HUKUM PERDATA \n(Burgerlijk Wetboek voor Indonesie) \n \nBUKU KESATU \nORANG \n \nBAB I \nMENIKMATI DAN KEHILANGAN HAK KEWARGAAN \n(Berlaku Bagi Golongan Timur Asing Bukan Tionghoa, dan Bagi Golongan \nTionghoa) \n \nPasal 1 \nMenikmati hak-hak kewargaan tidak tergantung pada hak-hak kenegaraan.  \n \nPasal 2 \nAnak yang ada dalam kandungan seorang perempuan dianggap telah lahir, setiap kali \nkepentingan si anak menghendakinya. Bila telah mati sewaktu dilahirkan, dia dianggap tidak \npernah ada.  \n \nPasal 3 \nTiada suatu hukuman pun yang mengakibatkan kematian perdata, atau hilangnya segala hak-\nhak kewargaan. \n \nBAB II \nAKTA-AKTA CATATAN SIPIL \n(Tidak Berlaku Bagi Golongan Timur Asing Bukan Tionghoa, dan Bagi Golongan \nTionghoa) \n \nBAGIAN 1 \nDaftar Catatan Sipil Pada Umumnya \n \nPasal 4 \nTanpa mengurangi ketentuan dalam Pasal 10 Ketentuan-ketentuan Umum Perundang-\nundangan di Indonesia, maka bagi golongan Eropa di

In [4]:
print(responses.pages[194].markdown + responses.pages[195].markdown + responses.pages[196].markdown)

Perdata; namun bila dilakukan di muka umum, penjualan itu harus dihadiri oleh para wali 
pengawas dan pengampu pengawas, atau setidak-tidaknya setelah mereka dipanggil 
secukupnya.  
Bila salah seorang dan para ahli waris membeli suatu barang tetap, maka hal itu mempunyai 
akibat yang sama terhadapnya seperti jika dia memperolehnya pada waktu pemisahan harta itu. 
  
Pasal 1077 
Penilaian barang-barang yang dalam harta peninggalan itu pada waktu dilaksanakan pemisahan 
harta peninggalan, diadakan sebagai berikut: 
Efek-efek, surat-surat piutang dan saham-saham dalam perusahaan-perusahaan, yang 
dicantumkan dalam berita-berita harga yang dibuat dan diumumkan secara resmi, dinilai 
menurut berita-berita harga itu.  
Barang-barang bergerak lainnya dinilai menurut harga taksiran pada waktu mengadakan 
pemerincian harta peninggalan itu, kecuali bila seorang ahli waris seorang atau lebih 
menghendaki diadakan penaksiran lebih lanjut oleh seorang ahli;  
Barang-barang tetap dinilai menurut ha

In [5]:
class KUHPerdataParser:
    def parse(self, text: str) -> dict:
        text = text.replace("\r\n", "\n").replace("\r", "\n")
        lines = [line.rstrip() for line in text.split("\n")]

        data = {"kitab": "", "keterangan": [], "buku": []}

        current_buku = None
        current_bab = None
        current_bagian = None
        current_pasal = None

        expect_title_for = None
        buffer_lines = []

        seen_pasal_numbers = set()  # <--- make pasal unique

        def is_heading(s: str) -> bool:
            s = s.strip()
            if not s:
                return False
            if re.match(r"^BUKU\b", s, re.IGNORECASE):
                return True
            if re.match(r"^BAB\b", s, re.IGNORECASE):
                return True
            if re.match(r"^BAGIAN\b", s, re.IGNORECASE):
                return True
            # pasal heading with extra text (not used for pasal start)
            if re.match(r"^Pasal\s+\d+[A-Za-z]*\s*[.:]?\s*$", s, re.IGNORECASE):
                return True
            return False

        def add_buffer_line(line: str) -> None:
            nonlocal buffer_lines
            if line == "":
                if buffer_lines and buffer_lines[-1] != "":
                    buffer_lines.append("")
                return

            if not buffer_lines:
                buffer_lines.append(line.strip())
                return

            prev = buffer_lines[-1]
            if prev == "":
                buffer_lines.append(line.strip())
                return

            if prev.endswith("-") and line and line.lstrip() and line.lstrip()[0].isalnum():
                buffer_lines[-1] = prev[:-1] + line.lstrip()
            else:
                buffer_lines.append(line.strip())

        def join_buffer_to_text() -> str:
            paragraphs = []
            current = []
            for ln in buffer_lines:
                if ln == "":
                    if current:
                        paragraphs.append(" ".join(current).strip())
                        current = []
                    continue
                current.append(ln)
            if current:
                paragraphs.append(" ".join(current).strip())

            joined = "\n\n".join([p for p in paragraphs if p])
            joined = re.sub(r"\s{2,}", " ", joined).strip()
            return joined

        def flush_pasal_text() -> None:
            nonlocal buffer_lines, current_pasal
            if current_pasal is None:
                buffer_lines = []
                return
            content = join_buffer_to_text()
            if content:
                if current_pasal["text"]:
                    current_pasal["text"] = (current_pasal["text"] + "\n\n" + content).strip()
                else:
                    current_pasal["text"] = content
            buffer_lines = []

        def ensure_buku() -> None:
            nonlocal current_buku
            if current_buku is None:
                current_buku = {"label": "", "judul": "", "catatan": [], "bab": [], "pasal": []}
                data["buku"].append(current_buku)

        def ensure_bab() -> None:
            nonlocal current_bab
            ensure_buku()
            if current_bab is None:
                current_bab = {"label": "", "judul": "", "catatan": [], "bagian": [], "pasal": []}
                current_buku["bab"].append(current_bab)

        def get_pasal_container():
            if current_bagian is not None:
                return current_bagian["pasal"]
            if current_bab is not None:
                return current_bab["pasal"]
            if current_buku is not None:
                return current_buku["pasal"]
            return None

        def attach_note(line: str) -> None:
            if current_bagian is not None:
                current_bagian["catatan"].append(line)
            elif current_bab is not None:
                current_bab["catatan"].append(line)
            elif current_buku is not None:
                current_buku["catatan"].append(line)
            else:
                data["keterangan"].append(line)

        for raw_line in lines:
            stripped = raw_line.strip()

            if stripped == "":
                if current_pasal is not None:
                    add_buffer_line("")
                continue

            if current_buku is None and not is_heading(stripped):
                if not data["kitab"]:
                    data["kitab"] = stripped
                else:
                    data["keterangan"].append(stripped)
                continue

            if expect_title_for is not None and not is_heading(stripped):
                if expect_title_for == "buku" and current_buku is not None:
                    current_buku["judul"] = stripped
                elif expect_title_for == "bab" and current_bab is not None:
                    current_bab["judul"] = stripped
                elif expect_title_for == "bagian" and current_bagian is not None:
                    current_bagian["judul"] = stripped
                expect_title_for = None
                continue

            m = re.match(r"^BUKU\s+(.+)$", stripped, re.IGNORECASE)
            if m:
                flush_pasal_text()
                buku_label = m.group(1).strip()
                current_buku = {"label": buku_label, "judul": "", "catatan": [], "bab": [], "pasal": []}
                data["buku"].append(current_buku)
                current_bab = None
                current_bagian = None
                current_pasal = None
                expect_title_for = "buku"
                continue

            m = re.match(r"^BAB\s+(.+)$", stripped, re.IGNORECASE)
            if m:
                flush_pasal_text()
                bab_label = m.group(1).strip()
                ensure_buku()
                current_bab = {"label": bab_label, "judul": "", "catatan": [], "bagian": [], "pasal": []}
                current_buku["bab"].append(current_bab)
                current_bagian = None
                current_pasal = None
                expect_title_for = "bab"
                continue

            m = re.match(r"^BAGIAN\s+(.+)$", stripped, re.IGNORECASE)
            if m:
                flush_pasal_text()
                bagian_label = m.group(1).strip()
                ensure_bab()
                current_bagian = {"label": bagian_label, "judul": "", "catatan": [], "pasal": []}
                current_bab["bagian"].append(current_bagian)
                current_pasal = None
                expect_title_for = "bagian"
                continue

            # IMPORTANT: strict pasal heading (only "Pasal <no>")
            m = re.match(r"^Pasal\s+(\d+)([A-Za-z]*)\s*[.:]?\s*$", stripped, re.IGNORECASE)
            if m:
                number_part = m.group(1)
                suffix_part = (m.group(2) or "").lower()   # normalize a/A -> a
                pasal_id = f"{number_part}{suffix_part}"   # "319", "319a", "319b", ...

                # uniqueness: if already parsed, treat as normal text (do not pivot)
                if pasal_id in seen_pasal_numbers:
                    if current_pasal is not None:
                        add_buffer_line(stripped)
                    else:
                        attach_note(stripped)
                    continue

                seen_pasal_numbers.add(pasal_id)
                flush_pasal_text()

                container = get_pasal_container()
                if container is None:
                    ensure_buku()
                    container = current_buku["pasal"]

                current_pasal = {"nomor": pasal_id, "text": ""}  # <-- store as string
                container.append(current_pasal)
                continue

            # catatan in parentheses right after titles
            if stripped.startswith("(") and stripped.endswith(")"):
                if current_bagian is not None and len(current_bagian["pasal"]) == 0 and current_bagian["judul"] != "":
                    current_bagian["catatan"].append(stripped)
                    continue
                if current_bab is not None and len(current_bab["pasal"]) == 0 and len(current_bab["bagian"]) == 0 and current_bab["judul"] != "":
                    current_bab["catatan"].append(stripped)
                    continue
                if current_buku is not None and len(current_buku["pasal"]) == 0 and len(current_buku["bab"]) == 0 and current_buku["judul"] != "":
                    current_buku["catatan"].append(stripped)
                    continue

            if current_pasal is not None:
                add_buffer_line(stripped)
            else:
                attach_note(stripped)

        flush_pasal_text()
        return data

    def to_json(self, parsed: dict, indent: int = 2) -> str:
        return json.dumps(parsed, ensure_ascii=False, indent=indent)

    def to_dataframe(self, parsed: dict) -> pd.DataFrame:
        rows = []

        def add_pasal_rows(pasal_list, buku_label, buku_judul, bab_label, bab_judul, bagian_label, bagian_judul):
            for p in pasal_list:
                rows.append({
                    "buku_label": buku_label,
                    "buku_judul": buku_judul,
                    "bab_label": bab_label,
                    "bab_judul": bab_judul,
                    "bagian_label": bagian_label,
                    "bagian_judul": bagian_judul,
                    "pasal_nomor": p.get("nomor"),
                    "pasal_text": p.get("text", "")
                })

        for buku in parsed.get("buku", []):
            buku_label = buku.get("label", "")
            buku_judul = buku.get("judul", "")

            add_pasal_rows(buku.get("pasal", []), buku_label, buku_judul, "", "", "", "")

            for bab in buku.get("bab", []):
                bab_label = bab.get("label", "")
                bab_judul = bab.get("judul", "")

                add_pasal_rows(bab.get("pasal", []), buku_label, buku_judul, bab_label, bab_judul, "", "")

                for bagian in bab.get("bagian", []):
                    bagian_label = bagian.get("label", "")
                    bagian_judul = bagian.get("judul", "")

                    add_pasal_rows(
                        bagian.get("pasal", []),
                        buku_label, buku_judul,
                        bab_label, bab_judul,
                        bagian_label, bagian_judul
                    )

        return pd.DataFrame(rows)


In [9]:
pymupdf_text = "\n".join([page.markdown for page in responses.pages])
parser = KUHPerdataParser()
parsed = parser.parse(pymupdf_text)
df = parser.to_dataframe(parsed)
df.to_csv(r"C:\Users\ghana\OneDrive\kuliah\Semester 8\TA\data\statute\kuh_perdata.csv", index=False)

In [ ]:
def extract_headings_sequential(text: str):
    pattern = re.compile(
        r"(?ms)"
        r"(^\s*(BAB)\s+([^\n]+)\s*$"
        r"|^\s*(BAGIAN)\s+(\d+)\s*$"
        r"|^\s*(Pasal)\s+(\d+)\b\s*$)"
    )

    def next_title(start_idx: int):
        rest = text[start_idx:]
        m = re.search(r"(?m)^[ \t]*(?!\s*$)(?!\s*\(.*\)\s*$)(.+)\s*$", rest)
        return m.group(1).strip() if m else ""

    out = []
    for m in pattern.finditer(text):
        end_pos = m.end(1)

        if m.group(2) == "BAB":
            nomor = m.group(3).strip()
            teks = next_title(end_pos)
            out.append(("bab", nomor, teks))

        elif m.group(4) == "BAGIAN":
            nomor = m.group(5).strip()
            teks = next_title(end_pos)
            out.append(("bagian", nomor, teks))

        else:  # Pasal
            nomor = m.group(7).strip()
            out.append(("pasal", nomor, ""))

    return out

In [ ]:

sections = extract_headings_sequential(pymupdf_text)
with open("sections.txt", "w", encoding="utf-8") as f:
    for typ, nomor, teks in sections:
        # format: type | nomor | teks
        f.write(f"{typ}\t{nomor}\t{teks}\n")